In [ ]:
########################################
#getting system arguments
import sys
def GetArg_dataName(default="Variables_Perturbations"):
    """
    #non-perturbations
    #Run One: python Tracked_Profiles.py Variables
    #Run Two: python Tracked_Profiles.py Entrainment
    #Run Three: python Tracked_Profiles.py PROCESSED_Entrainment (also can add _DivideMassFlux)
    #Run Four: python Tracked_Profiles.py W_Budgets
    #Run Five: python Tracked_Profiles.py QV_Budgets
    #Run Six: python Tracked_Profiles.py TH_Budgets

    #perturbations
    #Run One: python Tracked_Profiles.py Variables_Perturbations
    """
    # If run inside Jupyter, sys.argv will include ipykernel arguments
    if any("ipykernel_launcher" in arg for arg in sys.argv):
        print(f"Using default dataName: {default}")
        return default

    # If a user-specified argument exists, use it
    if len(sys.argv) > 1:
        out=sys.argv[1]
        print(f"Using argument dataName: {out}")
        return out

    return default

dataName = GetArg_dataName()

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracked_Profiles", dataName="Tracked_Profiles",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
##############################################
#JOB ARRAY

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def MakeDataDictionary(variableNames,t,printstatement=False):
    timeString = ModelData.timeStrings[t]
    # print(f"Getting data from {timeString}","\n")
    
    dataDictionary = {variableName: CallLagrangianArray(ModelData, DataManager, timeString, variableName=variableName, printstatement=printstatement) 
                      for variableName in variableNames}      
    return dataDictionary
    
def GetSpatialData(t):    
    variableNames = ['Z']
    dataDictionary = MakeDataDictionary(variableNames,t)
    [Z] = (dataDictionary[k] for k in variableNames)
    return Z

In [ ]:
####################################
#RUN SETUP

In [ ]:
#data variable list
def GetVarNames(dataName): 
    Processed_string = "PROCESSED_" if "PROCESSED_" in dataName else ""
    DivideMassFlux_string = "_DivideMassFlux" if "_DivideMassFlux" in dataName else ""

    
    if dataName=="Variables":
        varNames = ['W', 'QV','QCQI', 'RH_vapor', 'THETA_v', 'THETA_e', 'MSE', 'HMC', 'CONVERGENCE','VMF_g','VMF_c']
    elif dataName == "Entrainment":
        varNames = ['D_c', 'D_g', 'E_c', 'E_g',
                          'TransferD_c', 'TransferD_g', 
                          'TransferE_c', 'TransferE_g']
    elif dataName == f"{Processed_string}Entrainment{DivideMassFlux_string}":
        varNames = [f'{Processed_string}D{DivideMassFlux_string}_c', f'{Processed_string}D{DivideMassFlux_string}_g', 
                    f'{Processed_string}E{DivideMassFlux_string}_c', f'{Processed_string}E{DivideMassFlux_string}_g',
                    f'{Processed_string}TransferD{DivideMassFlux_string}_c', f'{Processed_string}TransferD{DivideMassFlux_string}_g', 
                    f'{Processed_string}TransferE{DivideMassFlux_string}_c', f'{Processed_string}TransferE{DivideMassFlux_string}_g']
    elif dataName=="W_Budgets":
        varNames = ['WB_BUOY', 'WB_HADV', 'WB_HIDIFF', 'WB_HTURB',
                    'WB_PGRAD', 'WB_VADV', 'WB_VIDIFF', 'WB_VTURB']
    elif dataName=="QV_Budgets":
        varNames = ['QVB_HADV', 'QVB_HIDIFF', 'QVB_HTURB', 'QVB_MP',
                    'QVB_VADV', 'QVB_VIDIFF', 'QVB_VTURB']
    elif dataName=="TH_Budgets":
        varNames = ['PTB_DISS', 'PTB_DIV', 'PTB_HADV', 'PTB_HIDIFF', 
                    'PTB_HTURB', 'PTB_MP', 'PTB_RAD', 'PTB_VADV', 
                    'PTB_VIDIFF', 'PTB_VTURB']

    elif dataName=="Variables_Perturbations":
        varNames = ["QV_prime","QCQI_prime","RH_vapor_prime",
                    "W_prime","VMF_g_prime",'VMF_c_prime',
                    "HMC_prime",
                    "THETA_v_prime",'THETA_e_prime','MSE_prime']
    return varNames

In [ ]:
########################################
#RUNNING FUNCTIONS

In [ ]:
#Functions for Initializing Profile Arrays
def CopyStructure(dictionary, placeholder=None):
    """Deep-copy dictionary structure, replacing leaves with a given placeholder."""
    if isinstance(dictionary, dict):
        return {k: CopyStructure(v, placeholder) for k, v in dictionary.items()}
    else:
        return placeholder

def InitializeProfileArrays(trackedArrays, varNames, zhs=ModelData.zh):
    """
    Create a new dictionary with the same nested structure as trackedArrays,
    and for each variable name, create:
        - 'profile_array' / 'profile_array_squares'
        - 'profile_array_left' / 'profile_array_left_squares'
        - 'profile_array_right' / 'profile_array_right_squares'
    Each array has shape (len(zhs), 3) and zhs in the last column.
    """
    profileArraysDictionary = {}

    for category, depth_dict in trackedArrays.items():  # e.g. 'CL', 'SBF'
        profileArraysDictionary[category] = {}

        for depth_type in depth_dict.keys():  # e.g. 'ALL', 'SHALLOW', 'DEEP'
            profileArraysDictionary[category][depth_type] = {}

            for varName in varNames:
                # Create base profile array
                base_profile = np.zeros((len(zhs), 3))
                base_profile[:, 2] = zhs

                profileArraysDictionary[category][depth_type][varName] = {
                    # Main / all parcels
                    "profile_array": base_profile.copy(),
                    "profile_array_squares": base_profile.copy(),

                    # Left subset (-1)
                    "profile_array_left": base_profile.copy(),
                    "profile_array_left_squares": base_profile.copy(),

                    # Right subset (+1)
                    "profile_array_right": base_profile.copy(),
                    "profile_array_right_squares": base_profile.copy(),
                }
    return profileArraysDictionary

In [ ]:
def GetParcelNumbers(trackedArray, t):
    """
    Return all parcel indices (p) and their corresponding row indices
    for parcels that are active at time t.
    Vectorized, no row-by-row loops.
    """
    t_start = trackedArray[:, 1]
    t_end   = np.minimum(trackedArray[:, 2] + trackedArray[:, 3], ModelData.Ntime)

    # Boolean mask for rows active at time t
    mask = (t >= t_start) & (t <= t_end)

    # Extract parcel numbers and their corresponding row indices
    selectedRows = np.where(mask)[0]
    selectedPs = trackedArray[selectedRows, 0]
    leftRightIndexes = trackedArray[selectedRows, 4]

    return selectedRows, selectedPs, leftRightIndexes

In [ ]:
def MakeTrackedProfiles(trackedArrays,profileArraysDictionary,varNames,VARs,Z,t):
    """
    Update profileArraysDictionary with variable data for parcels active at time t.
    Accumulates sums and counts in both profile_array and profile_array_squares.
    """
    #CALCULATING
    for key1, subdict in trackedArrays.items():         # e.g. 'CL', 'SBF'
        # print("\t",f'working on {key1}')
        for key2, trackedArray in subdict.items():           # e.g. 'ALL', 'DEEP'
            # print("\t\t",f'working on {key2}')
            #loading the profile array to fill
            profileArray = profileArraysDictionary[key1][key2] 
    
            #getting parcels in trackedArray to run through
            _, selectedPs, leftRightIndexes = GetParcelNumbers(trackedArray, t) #get parcels that are counted at time t
            
            #getting Z data
            zLevels = Z[selectedPs]

            # Boolean masks
            mask_left = leftRightIndexes == -1
            mask_right = leftRightIndexes == 1
            
            for varName in varNames:
                #getting data
                masked_data = VARs[varName][selectedPs]
                
                NaN_mask = ~np.isnan(masked_data)
                masked_data_valid = masked_data[NaN_mask]
                zLevels_valid = zLevels[NaN_mask]
                
                 # --- MAIN appending (all parcels go here) ---
                np.add.at(profileArray[varName]["profile_array"][:, 0], zLevels_valid, masked_data_valid)
                np.add.at(profileArray[varName]["profile_array"][:, 1], zLevels_valid, 1)
                np.add.at(profileArray[varName]["profile_array_squares"][:, 0], zLevels_valid, masked_data_valid**2)
                np.add.at(profileArray[varName]["profile_array_squares"][:, 1], zLevels_valid, 1)
                
                # --- LEFT subset (-1 only) ---
                if np.any(mask_left):
                    mask_left_valid = mask_left[NaN_mask]
                    np.add.at(profileArray[varName]["profile_array_left"][:, 0], zLevels_valid[mask_left_valid], masked_data_valid[mask_left_valid])
                    np.add.at(profileArray[varName]["profile_array_left"][:, 1], zLevels_valid[mask_left_valid], 1)
                    np.add.at(profileArray[varName]["profile_array_left_squares"][:, 0], zLevels_valid[mask_left_valid], masked_data_valid[mask_left_valid]**2)
                    np.add.at(profileArray[varName]["profile_array_left_squares"][:, 1], zLevels_valid[mask_left_valid], 1)

                # --- RIGHT subset (+1 only) ---
                if np.any(mask_right):
                    mask_right_valid = mask_right[NaN_mask]
                    np.add.at(profileArray[varName]["profile_array_right"][:, 0], zLevels_valid[mask_right_valid], masked_data_valid[mask_right_valid])
                    np.add.at(profileArray[varName]["profile_array_right"][:, 1], zLevels_valid[mask_right_valid], 1)
                    np.add.at(profileArray[varName]["profile_array_right_squares"][:, 0], zLevels_valid[mask_right_valid], masked_data_valid[mask_right_valid]**2)
                    np.add.at(profileArray[varName]["profile_array_right_squares"][:, 1], zLevels_valid[mask_right_valid], 1)

    return profileArraysDictionary

In [ ]:
def RunCode():
    for t in tqdm(loop_elements, desc="Processing"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
        print("#" * 40,"\n",f"Processing timestep {t}/{loop_elements[-1]}")
        timeString = ModelData.timeStrings[t]
    
        #Forming Dictionary for Profile Arrays for current timestep
        trackedProfileArrays = CopyStructure(trackedArrays)
        profileArraysDictionary = InitializeProfileArrays(trackedProfileArrays,varNames)
        
        #getting variable data
        VARs = MakeDataDictionary(varNames, t)
        Z = GetSpatialData(t)
    
        #making tracked profiles
        profileArraysDictionary = MakeTrackedProfiles(trackedArrays,profileArraysDictionary,varNames,VARs,Z,t)
    
        #saving tracked profiles for current timestep
        TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, profileArraysDictionary, dataName, t)
    return profileArraysDictionary

In [ ]:
########################################
#RUNNING
running=True #KEEP TRUE WHEN JOBARRAY IS RUNNING
# running=False 

In [ ]:
if running:
    #Loading in Tracked Parcels Info
    trackedArrays,LevelsDictionary = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,
                                                             Results_InputOutput_Class)
    
    #Removing After Ascent Count for SHALLOW parcels
    #Reason is there is a lot of shallow parcels
    #that hit their peak below 4 km, 
    #but stay in the cloud in turbulent eddies and later exit at much higher levels
    #and exit the cloudy updraft higher
    # for key1, subdict in trackedArrays.items():
    #     subdict['SHALLOW'][:,3]=0
    #testing without for now #*
    
    # Get Variable Names
    varNames = GetVarNames(dataName)

In [ ]:
if running:
    profileArraysDictionary = RunCode()

In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
recombine=True

In [ ]:
def RecombineProfiles(ModelData, DataManager, tIndicies, dataName):
    """
    Combine tracked profiles across all timesteps using the first as a template.
    """
    print(f"Recombining {ModelData.Ntime} TrackedProfiles files...\n")

    trackedProfileArrays = None

    for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
        profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

        if t == tIndicies[0]:
            # Deep copy structure so we don’t overwrite the first timestep’s data
            trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
            continue  # move to next time step — skip accumulation for t=0

        # Add all later times to the initial one
        for key1 in profileArraysDictionary:
            for key2 in profileArraysDictionary[key1]:
                for varName in profileArraysDictionary[key1][key2]:
                    for arrayName in ["profile_array", "profile_array_squares",
                                      "profile_array_left", "profile_array_left_squares",
                                      "profile_array_right", "profile_array_right_squares"]:
                        trackedProfileArrays[key1][key2][varName][arrayName][:, 0:2] += (
                            profileArraysDictionary[key1][key2][varName][arrayName][:, 0:2]
                        )
    return trackedProfileArrays

In [ ]:
if recombine==True:
    timeStrings_datetime = TrackedProfiles_DataLoading_CLASS.ConvertTimeStringsToDatetime(ModelData.timeStrings)
    timeSlice_all = np.arange(ModelData.Ntime)
    timeSlice_1 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,10,12) #late morning / pre-convective
    timeSlice_2 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,14) #CBL growth
    timeSlice_3 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,14,17) #mature convection / early decay

    ################################################################

    timeSlice_4 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,10,11) #SBF initiation 1
    timeSlice_5 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,11,12) #SBF initiation 2
    timeSlice_6 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,13) #CBL growth 1
    timeSlice_7 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,13,14) #CBL growth 2


    timeSlices = [timeSlice_all,
                  timeSlice_1, timeSlice_2, timeSlice_3,
                  timeSlice_4, timeSlice_5, timeSlice_6, timeSlice_7]
    timeLabels = ['combined',
                  'combined_10-12', 'combined_12-14', 'combined_14-17',
                  'combined_10-11', 'combined_11-12', 'combined_12-13', 'combined_13-14']

In [ ]:
if recombine==True:
    # for dataName in ["Variables",
    #                  "Entrainment","PROCESSED_Entrainment","PROCESSED_Entrainment_DivideMassFlux",
    #                  "W_Budgets","QV_Budgets","TH_Budgets"]:
    for dataName in ['Variables_Perturbations']:

        for timeSlice, timeLabel in zip(timeSlices, timeLabels): #looping through above time windows
            trackedProfileArrays = RecombineProfiles(ModelData, DataManager_TrackedProfiles,tIndicies=timeSlice,dataName=dataName)
            TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, trackedProfileArrays, dataName, t=timeLabel)

In [ ]:
##############################################
#LOADING BACK IN

In [ ]:
##############################################
#TESTING

In [ ]:
# #testing calculation of cloudbase using profile min qcqi satisfying threshold

# #Loading in Tracked Parcels Info
# trackedArrays,LevelsDictionary = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,
#                                                          Results_InputOutput_Class)
# LFC_profile = LevelsDictionary['LFC_profile'].copy()
# LCL_profile = LevelsDictionary['LCL_profile'].copy()

# timeStrings_datetime = TrackedProfiles_DataLoading_CLASS.ConvertTimeStringsToDatetime(ModelData.timeStrings)
# timeSlice_all = np.arange(ModelData.Ntime)
# timeSlice_1 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,10,12) #late morning / pre-convective
# timeSlice_2 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,14) #CBL growth
# timeSlice_3 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,14,17) #mature convection / early decay

# ################################################################

# timeSlice_4 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,10,11) #SBF initiation 1
# timeSlice_5 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,11,12) #SBF initiation 2
# timeSlice_6 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,13) #CBL growth 1
# timeSlice_7 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,13,14) #CBL growth 2


# timeSlices = [timeSlice_all,
#               timeSlice_1, timeSlice_2, timeSlice_3,
#               timeSlice_4, timeSlice_5, timeSlice_6, timeSlice_7]
# timeLabels = ['combined',
#               'combined_10-12', 'combined_12-14', 'combined_14-17',
#               'combined_10-11', 'combined_11-12', 'combined_12-13', 'combined_13-14']

# def GetCloudBaseTimeseries(ModelData, DataManager, tIndicies, #depreciated, there isn't a profile for QC so CB estimate is overestimated
#                            dataName="Variables"):
#     """
#     Combine tracked profiles across all timesteps using the first as a template.
#     """
#     print(f"Recombining {ModelData.Ntime} TrackedProfiles files...\n")

#     trackedProfileArrays = None

#     CB_profile=[];CB_times=[]
#     for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
#         if "Entrainment" in dataName and t == ModelData.Ntime-1:
#             continue
#         profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

#         # Add all later times to the initial one
#         varName="QCQI"
#         arrayName = "profile_array"
#         for key1 in profileArraysDictionary:
#             for key2 in profileArraysDictionary[key1]:
#                 profile=profileArraysDictionary[key1][key2][varName][arrayName][:, 0]
#                 where=np.where(profile>1e-6)
#                 if where[0].size > 0:
#                     CB_profile.append(ModelData.zh[where[0][0]])
#                     CB_times.append(t)
#                 else:
#                     CB_profile.append(np.nan)
#                     CB_times.append(np.nan)
#     return CB_profile,CB_times

# [CB_profile,CB_times]=GetCloudBaseTimeseries(ModelData, DataManager_TrackedProfiles,tIndicies=timeSlices[0])

# def AddAt(profile_mean,profile_mean_count,
#           profile,times):
#     np.add.at(profile_mean, times, profile)
#     np.add.at(profile_mean_count, times, 1)
#     return profile_mean,profile_mean_count
# def NanDivide(a, b):
#     return np.divide(a, b, where=b!=0, out=np.full_like(a, np.nan, dtype=float))
# CB_profile_mean = np.full(ModelData.Ntime,0,dtype='float')
# CB_profile_mean_count = np.full(ModelData.Ntime,0,dtype='int')
# CB_profile = [a for a in CB_profile if ~np.isnan(a)]
# CB_times = [a for a in CB_times if ~np.isnan(a)]
# AddAt(CB_profile_mean,CB_profile_mean_count,
#       CB_profile,CB_times)
# CB_profile_mean = NanDivide(CB_profile_mean,CB_profile_mean_count)

# from scipy.ndimage import gaussian_filter1d
# smooth = CB_profile_mean.copy()
# where=np.where(~np.isnan(CB_profile_mean))[0]
# CB_smooth = gaussian_filter1d(CB_profile_mean[where], sigma=2)
# smooth[where]=CB_smooth

# plt.plot(np.arange(ModelData.Ntime),LFC_profile,'green')
# plt.plot(np.arange(ModelData.Ntime),LCL_profile,'red')
# plt.plot(np.arange(ModelData.Ntime),CB_profile_mean,'blue')
# plt.plot(np.arange(ModelData.Ntime),smooth,'k')

In [ ]:
# #testing calculating average cloud base with eulerian data

# t=100
# timeString = ModelData.timeStrings[t]
# qc = DataManager.GetTimestepData(DataManager.inputDataDirectory, timeString, variableName="qc")


# qc_thresh = 1e-6
# # cloud mask
# mask = qc >= qc_thresh  # (z, y, x)
# # columns that contain cloud
# has_cloud = np.any(mask, axis=0)  # (y, x)
# # first cloudy level index
# first_idx = np.argmax(mask, axis=0)  # (y, x)
# # initialize output
# cloud_base = np.full(first_idx.shape, np.nan, dtype=float)
# # assign cloud base height
# cloud_base[has_cloud] = ModelData.zh[first_idx[has_cloud]]


# # plt.plot(np.mean(qc,axis=(1,2)),ModelData.zh)
# # ModelData.zh[np.where(np.mean(qc,axis=(1,2))>1e-6)[0][0]]

# plt.contourf(cloud_base)
# plt.colorbar()

# np.nanmin(cloud_base)

In [ ]:
# #testing: comparing above vectorized version to multi-loop version below

# #pure calculation method (slow)
# t=80

# a = trackedArrays['CL']['DEEP']
# ps = a[:,0]
# t1s = a[:,1]
# t2s = a[:,2]+a[:,3]

# profile=np.zeros((ModelData.Nzh,3))
# profile[:,2] = ModelData.zh

# plen = len(ps)
# for count, (p,t1,t2) in enumerate(zip(ps,t1s,t2s)):
#     if count % 100 == 0: print(f"{count}/{plen}")
#     ts = np.arange(t1, t2+1)
    
#     for t_ in ts:
#         if t_ != t:
#             continue
#         Z = GetSpatialData(t)[p]
#         var = MakeDataDictionary(["W"], t=t)["W"][p]
#         profile[Z,0]+=var
#         profile[Z,1]+=1


# #plotting

# def Plot(a,color):
#     b= a[:,0]/a[:,1]
#     # b*=1000
#     plt.plot(b,a[:,2],color=color)
# one = profileArraysDictionary['CL']['DEEP']['W']['profile_array'] #created using MakeTrackedProfiles function
# two = profile.copy()
# Plot(one,color='black')
# Plot(two,color='blue') #plots are the same
# np.all(one[:,0]==two[:,0]) #=True

In [ ]:
# #testing affect on qv of averaging over specific timesteps

# sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
# from CLASSES_TrackedProfiles import TrackedProfiles_Plotting_CLASS

# def DoSum(ModelData, DataManager, tIndicies):
#     dataName='Variables';arrayName='profile_array_right'
#     trackedProfileArrays = None
#     for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
#         # if "Entrainment" in dataName and t == ModelData.Ntime-1:
#         #     continue
#         profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

#         if t == tIndicies[0]:
#             # Deep copy structure so we don’t overwrite the first timestep’s data
#             trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
#             continue  # move to next time step — skip accumulation for t=0

#         # Add all later times to the initial one
#         for key1 in profileArraysDictionary:
#             for key2 in profileArraysDictionary[key1]:
#                 for varName in profileArraysDictionary[key1][key2]:
#                         trackedProfileArrays[key1][key2][varName][arrayName][:, 0:2] += (
#                             profileArraysDictionary[key1][key2][varName][arrayName][:, 0:2]
#                         )
#     return trackedProfileArrays

# # def DoSum(ModelData, DataManager, tIndicies):
# #     dataName='Variables';arrayName='profile_array_right'
# #     trackedProfileArrays = None
# #     for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
# #         # if "Entrainment" in dataName and t == ModelData.Ntime-1:
# #         #     continue
# #         profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

# #         if t == tIndicies[0]:
# #             # Deep copy structure so we don’t overwrite the first timestep’s data
# #             trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
# #             continue  # move to next time step — skip accumulation for t=0

# #         # Add all later times to the initial one
# #         for key1 in profileArraysDictionary:
# #             for key2 in profileArraysDictionary[key1]:
# #                 for varName in profileArraysDictionary[key1][key2]:
# #                     arr = profileArraysDictionary[key1][key2][varName][arrayName]
# #                     mean_t = SafeDivide(arr[:,0],arr[:,1])
# #                     validMask = np.isfinite(mean_t)
# #                     trackedProfileArrays[key1][key2][varName][arrayName][:, 0] += np.where(validMask, mean_t, 0.0)
# #                     trackedProfileArrays[key1][key2][varName][arrayName][:, 1] += validMask
# #     return trackedProfileArrays


# def MakePlot(axis,profile,color):
#     avg = TrackedProfiles_Plotting_CLASS.ProfileMean(profile)
#     multiplier=1000
#     x = multiplier * avg[:, 0]
#     y = avg[:, 1]
    
#     axis.plot(x,y,color=color)
#     axis.set_ylim(0,3)
#     axis.set_xlabel('qv (g/kg)'); axis.set_ylabel('z (km)')
#     axis.set_xlim(0,17)

# def MakePlots(timeSlice):
#     fig,axis=plt.subplots()
#     profiles = DoSum(ModelData, DataManager_TrackedProfiles, tIndicies=timeSlice)
#     shallow = profiles['CL']['SHALLOW']['QV']['profile_array_right']
#     deep = profiles['CL']['DEEP']['QV']['profile_array_right']
#     MakePlot(axis,shallow,'green')
#     MakePlot(axis,deep,'blue')
#     return fig

In [ ]:
# ########################
# timeSlice_alltimes = np.arange(ModelData.Ntime)
# fig = MakePlots(timeSlice=timeSlice_alltimes)
# fig.suptitle('RIGHT All Times')

# ########################
# timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,13) #CBL growth 2
# fig = MakePlots(timeSlice=timeSlice_afternoon)
# fig.suptitle('RIGHT 12-13')

# timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,13,14) #CBL growth 2
# fig = MakePlots(timeSlice=timeSlice_afternoon)
# fig.suptitle('RIGHT 13-14')

# timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,14,15) #CBL growth 2
# fig = MakePlots(timeSlice=timeSlice_afternoon)
# fig.suptitle('RIGHT 14-15')

# timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,15,16) #CBL growth 2
# fig = MakePlots(timeSlice=timeSlice_afternoon)
# fig.suptitle('RIGHT 15-16')

# timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,16,17) #CBL growth 2
# fig = MakePlots(timeSlice=timeSlice_afternoon)
# fig.suptitle('RIGHT 16-17')

# # timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,14) #CBL growth 2
# # fig = MakePlots(timeSlice=timeSlice_afternoon)
# # fig.suptitle('RIGHT 12-14')

# # timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,15) #CBL growth 2
# # fig = MakePlots(timeSlice=timeSlice_afternoon)
# # fig.suptitle('RIGHT 12-15')

# # timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,16) #CBL growth 2
# # fig = MakePlots(timeSlice=timeSlice_afternoon)
# # fig.suptitle('RIGHT 12-16')

# # timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,17) #CBL growth 2
# # fig = MakePlots(timeSlice=timeSlice_afternoon)
# # fig.suptitle('RIGHT 12-17')

# ###############################

# timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,10,11) #CBL growth 2
# fig = MakePlots(timeSlice=timeSlice_afternoon)
# fig.suptitle('RIGHT 10-11')

# timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,11,12) #CBL growth 2
# fig = MakePlots(timeSlice=timeSlice_afternoon)
# fig.suptitle('RIGHT 11-12')

# timeSlice_afternoon = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,10,12) #CBL growth 2
# fig = MakePlots(timeSlice=timeSlice_afternoon)
# fig.suptitle('RIGHT 10-12')

In [ ]:
# #old
# #testing affect on qv of averaging over specific timesteps

# sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
# from CLASSES_TrackedProfiles import TrackedProfiles_Plotting_CLASS
# import warnings

# def ReplaceDictionaryArrays(storageArrays,tIndicies):
#     for key1 in storageArrays:
#         for key2 in storageArrays[key1]:
#             for varName in storageArrays[key1][key2]:
#                 for arrName in storageArrays[key1][key2][varName]:
#                     storageArrays[key1][key2][varName][arrName] = np.full((len(tIndicies), ModelData.Nzh), np.nan)
#     return storageArrays

# def GetArray(ModelData, DataManager, tIndicies):
#     dataName='Variables';arrayName='profile_array_right'

#     trackedProfileArrays = None
#     for count, t in enumerate(tqdm(tIndicies, desc="Combining Profiles", unit="timestep")):
#         # if "Entrainment" in dataName and t == ModelData.Ntime-1:
#         #     continue
#         profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

#         # if t == tIndicies[0]:
#         if count==0:
#             # Deep copy structure so we don’t overwrite the first timestep’s data
#             storageArrays = copy.deepcopy(profileArraysDictionary)
#             storageArrays = ReplaceDictionaryArrays(storageArrays,tIndicies)
#             storageArrays_count = copy.deepcopy(profileArraysDictionary)
#             storageArrays_count = ReplaceDictionaryArrays(storageArrays_count,tIndicies)
#             continue  # move to next time step — skip accumulation for t=0
        
#         # Add all later times to the initial one
#         for key1 in profileArraysDictionary:
#             for key2 in profileArraysDictionary[key1]:
#                 for varName in profileArraysDictionary[key1][key2]:
#                     storageArrays[key1][key2][varName][arrayName][count,:] = \
#                     profileArraysDictionary[key1][key2][varName][arrayName][:, 0]
#                     storageArrays_count[key1][key2][varName][arrayName][count,:] = \
#                     profileArraysDictionary[key1][key2][varName][arrayName][:, 1]
#     return storageArrays,storageArrays_count

# def SafeDivide(numerator, denominator):
#     return np.divide(
#         numerator,
#         denominator,
#         out=np.full_like(numerator, np.nan, dtype=float),
#         where=(denominator != 0)
#     )
    
# def MakePlots(storageArrays,storageArrays_count,takeCumSum=False):
    
#     fig, axes = plt.subplots(2, 2, figsize=(10, 10), constrained_layout=True)
    
#     # --- SHALLOW ---
#     a1 = storageArrays['CL']['SHALLOW']['QV']['profile_array_right']
#     a2 = storageArrays_count['CL']['SHALLOW']['QV']['profile_array_right']
#     if takeCumSum == True:
#         a1 = np.nancumsum(a1,axis=0);a2 = np.nancumsum(a2,axis=0) 
#     c1 = SafeDivide(a1, a2)
#     c1 *= 1000
    
#     # --- DEEP ---
#     a1 = storageArrays['CL']['DEEP']['QV']['profile_array_right']
#     a2 = storageArrays_count['CL']['DEEP']['QV']['profile_array_right']
#     if takeCumSum == True:
#             a1 = np.nancumsum(a1,axis=0);a2 = np.nancumsum(a2,axis=0) 
#     c2 = SafeDivide(a1, a2)
#     c2 *= 1000
    
#     # --- DIFFERENCE ---
#     cdiff = c1 - c2
    
#     # --- Shared scale for absolute plots ---
#     vmin = np.nanmin([c1, c2])
#     vmax = np.nanmax([c1, c2])
#     levels = np.linspace(0, vmax, 20)
    
#     # --- Symmetric scale for difference ---
#     # vmax_diff = np.nanmax(np.abs(cdiff))
#     vmax_diff = np.nanpercentile(np.abs(cdiff), 90)
#     levels_diff = np.linspace(-vmax_diff, vmax_diff, 15)
    
#     # ===== LEFT COLUMN =====
#     # SHALLOW
#     axis=axes[0, 0]
#     cf1 = axis.contourf(tIndicies/60+6, ModelData.zh, c1.T, levels=levels, cmap='turbo')
#     axis.axhline(4,color='black')
#     axis.set_title('Shallow')
#     axis.set_ylim(0, 6)
#     axis.set_ylabel('z (km)')
#     axis.set_xlabel('Time (hr)')
#     axis.set_xlim(left=10)
#     fig.colorbar(cf1, ax=axis).set_label('qv (g/kg)')
    
#     # DEEP
#     axis = axes[1, 0]
#     cf2 = axis.contourf(tIndicies/60+6, ModelData.zh, c2.T, levels=levels, cmap='turbo')
#     axis.axhline(4,color='black')
#     axis.set_title('Deep')
#     axis.set_ylim(0, 6)
#     axis.set_ylabel('z (km)')
#     axis.set_xlabel('Time (hr)')
#     axis.set_xlim(left=10)
#     fig.colorbar(cf2, ax=axis).set_label('qv (g/kg)')
    
#     # ===== RIGHT COLUMN =====
#     # SHALLOW - DEEP (top)
#     axis = axes[0, 1]
#     cf3 = axis.contourf(tIndicies/60+6, ModelData.zh, cdiff.T, levels=levels_diff, cmap="RdBu_r",extend='both')
#     axis.axhline(4,color='black')
#     axis.set_title('Shallow - Deep')
#     axis.set_ylim(0, 6)
#     axis.set_xlabel('Time (hr)')
#     axis.set_ylabel('z (km)')
#     axis.set_xlim(left=10)
#     fig.colorbar(cf3, ax=axis).set_label('Δqv (g/kg)')
        
#     axis = axes[1, 1]
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore", category=RuntimeWarning)
#         d1 = np.nanmean(c1, axis=0)
#         d2 = np.nanmean(c2, axis=0)
#     axis.plot(d1,ModelData.zh,color='green')
#     axis.plot(d2,ModelData.zh,color='blue')
#     axis.axhline(4,color='black')
#     axis.set_ylim(0,6)
#     axis.set_xlabel('qv (g/kg)')
#     axis.set_ylabel('z (km)')
#     axis.set_title('Shallow & Deep - Profiles')

# tIndicies = np.arange(ModelData.Ntime)
# [storageArrays,storageArrays_count] = GetArray(ModelData, DataManager_TrackedProfiles, tIndicies)

# MakePlots(storageArrays,storageArrays_count,takeCumSum=True)
# MakePlots(storageArrays,storageArrays_count,takeCumSum=False)

In [ ]:
# #old
# #testing unweighted average
# def RecombineProfiles(ModelData, DataManager, tIndicies):
#     """
#     Combine tracked profiles across all timesteps using the first as a template.
#     """
#     print(f"Recombining {ModelData.Ntime} TrackedProfiles files...\n")

#     trackedProfileArrays = None

#     for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
#         if "Entrainment" in dataName and t == ModelData.Ntime-1:
#             continue
#         profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

#         if t == tIndicies[0]:
#             # Deep copy structure so we don’t overwrite the first timestep’s data
#             trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
#             continue  # move to next time step — skip accumulation for t=0

#         # Add all later times to the initial one
#         for key1 in profileArraysDictionary:
#             for key2 in profileArraysDictionary[key1]:
#                 for varName in profileArraysDictionary[key1][key2]:
#                     for arrayName in ["profile_array", "profile_array_squares",
#                                       "profile_array_left", "profile_array_left_squares",
#                                       "profile_array_right", "profile_array_right_squares"]:
#                         arr = profileArraysDictionary[key1][key2][varName][arrayName]
#                         mean_t = SafeDivide(arr[:,0],arr[:,1])
#                         validMask = np.isfinite(mean_t)
#                         trackedProfileArrays[key1][key2][varName][arrayName][:, 0] += np.where(validMask, mean_t, 0.0)
#                         trackedProfileArrays[key1][key2][varName][arrayName][:, 1] += validMask
#     return trackedProfileArrays

# def SafeDivide(numerator, denominator):
#     return np.divide(
#         numerator,
#         denominator,
#         out=np.full_like(numerator, np.nan, dtype=float),
#         where=(denominator != 0)
#     )